In [50]:
import xml.etree.ElementTree as ET
import copy

In [ ]:
persontrips_filepath = "D:\\Dokumente\\sumo\\maloja\\11_hotel\\persontrips_maloja_hotel_offseason.xml"
persontrips_filepath_output = "D:\\Dokumente\\sumo\\maloja\\11_hotel\\persontrips_maloja_hotel_offseason_modded.xml"
routes_filepath = "D:\\Dokumente\\sumo\\maloja\\11_hotel\\persontrips_maloja_hotel_resolved_offseason_bus.rou.xml"
routes_filepath_output = "D:\\Dokumente\\sumo\\maloja\\11_hotel\\persontrips_maloja_hotel_resolved_offseason_bus_modded.rou.xml"

In [52]:
demand_arriving_leaving = {
    "taz_St_Moritz_Sink-taz_Bregaglia": 2,
    "taz_St_Moritz_Sink-taz_Silvaplana": 9,
    "taz_St_Moritz_Sink-taz_Sils_Engadin": 5,
    "taz_St_Moritz_Sink-taz_St_Moritz": 87
}

demand_activities = {
    "taz_Bregaglia-taz_Bregaglia_Summer": 1,
    "taz_Bregaglia-taz_Sils_Engadin_Summer": 1,
    "taz_Bregaglia-taz_Silvaplana_Summer": 1,
    "taz_Bregaglia-taz_St_Moritz_Summer": 1,
    
    "taz_Bregaglia-taz_St_Moritz": 1,
    "taz_Bregaglia-taz_Bregaglia": 1,
    "taz_Bregaglia-taz_Silvaplana": 1,
    "taz_Bregaglia-taz_Sils_Engadin": 1,
    "taz_Bregaglia-taz_St_Moritz_Sink": 1,

    "taz_Silvaplana-taz_Bregaglia_Summer": 1,
    "taz_Silvaplana-taz_Sils_Engadin_Summer": 1,
    "taz_Silvaplana-taz_Silvaplana_Summer": 1,
    "taz_Silvaplana-taz_St_Moritz_Summer": 1,

    "taz_Silvaplana-taz_St_Moritz": 2,
    "taz_Silvaplana-taz_Bregaglia": 2,
    "taz_Silvaplana-taz_Silvaplana": 2,
    "taz_Silvaplana-taz_Sils_Engadin": 2,
    "taz_Silvaplana-taz_St_Moritz_Sink": 2,
    
    "taz_Sils_Engadin-taz_Bregaglia_Summer": 1,
    "taz_Sils_Engadin-taz_Sils_Engadin_Summer": 1,
    "taz_Sils_Engadin-taz_Silvaplana_Summer": 1,
    "taz_Sils_Engadin-taz_St_Moritz_Summer": 1,

    "taz_Sils_Engadin-taz_St_Moritz": 1,
    "taz_Sils_Engadin-taz_Bregaglia": 1,
    "taz_Sils_Engadin-taz_Silvaplana": 1,
    "taz_Sils_Engadin-taz_Sils_Engadin": 1,
    "taz_Sils_Engadin-taz_St_Moritz_Sink": 1,
    
    "taz_St_Moritz-taz_Bregaglia_Summer": 6,
    "taz_St_Moritz-taz_Sils_Engadin_Summer": 6,
    "taz_St_Moritz-taz_Silvaplana_Summer": 6,
    "taz_St_Moritz-taz_St_Moritz_Summer": 6,

    "taz_St_Moritz-taz_St_Moritz": 11,
    "taz_St_Moritz-taz_Bregaglia": 11,
    "taz_St_Moritz-taz_Silvaplana": 11,
    "taz_St_Moritz-taz_Sils_Engadin": 11,
    "taz_St_Moritz-taz_St_Moritz_Sink": 11,
}

In [53]:
demand_hotel_arriving = copy.deepcopy(demand_arriving_leaving)
demand_hotel_leaving = copy.deepcopy(demand_arriving_leaving)

In [54]:
demand_activities_from_hotel = copy.deepcopy(demand_activities)
demand_activities_to_hotel = copy.deepcopy(demand_activities)

In [55]:
persontrips_file_xmltree = ET.parse(persontrips_filepath)
persontrips_file_xmlroot = persontrips_file_xmltree.getroot()

In [56]:
routes_file_xmltree = ET.parse(routes_filepath)
routes_file_xmlroot = routes_file_xmltree.getroot()

In [57]:
for trip_child in list(routes_file_xmlroot):
    has_ride = False
    for stage_child in list(trip_child):
        if stage_child.tag == "ride":
            if "intended" in stage_child.attrib:
                has_ride = True
                break

    if has_ride == True:
        for persontrip_child in list(persontrips_file_xmlroot):
            if trip_child.get("id") == persontrip_child.get("id"):
                demand_ft_key = str(persontrip_child[0].get("fromTaz")) + "-" + str(persontrip_child[0].get("toTaz"))
                demand_tf_key = str(persontrip_child[0].get("toTaz")) + "-" + str(persontrip_child[0].get("fromTaz"))

                if 36000 <= float(str(trip_child.get("depart"))) <= 43200 and demand_tf_key in demand_hotel_leaving and demand_hotel_leaving[demand_tf_key] > 0:
                    demand_hotel_leaving[demand_tf_key] = demand_hotel_leaving[demand_tf_key] - 1
                    break
                elif 43200 <= float(str(trip_child.get("depart"))) <= 50400 and demand_ft_key in demand_hotel_arriving and demand_hotel_arriving[demand_ft_key] > 0:
                    demand_hotel_arriving[demand_ft_key] = demand_hotel_arriving[demand_ft_key] - 1
                    break
                elif demand_ft_key in demand_activities_from_hotel and demand_activities_from_hotel[demand_ft_key] > 0:
                    demand_activities_from_hotel[demand_ft_key] = demand_activities_from_hotel[demand_ft_key] - 1
                    break
                elif demand_tf_key in demand_activities_to_hotel and demand_activities_to_hotel[demand_tf_key] > 0:
                    demand_activities_to_hotel[demand_tf_key] = demand_activities_to_hotel[demand_tf_key] - 1
                    break
                else:
                    routes_file_xmlroot.remove(trip_child)
                    break

    if has_ride == False:
        routes_file_xmlroot.remove(trip_child)

In [58]:
demand_hotel_leaving

{'taz_St_Moritz_Sink-taz_Bregaglia': 0,
 'taz_St_Moritz_Sink-taz_Silvaplana': 0,
 'taz_St_Moritz_Sink-taz_Sils_Engadin': 0,
 'taz_St_Moritz_Sink-taz_St_Moritz': 0}

In [59]:
demand_hotel_arriving

{'taz_St_Moritz_Sink-taz_Bregaglia': 0,
 'taz_St_Moritz_Sink-taz_Silvaplana': 0,
 'taz_St_Moritz_Sink-taz_Sils_Engadin': 0,
 'taz_St_Moritz_Sink-taz_St_Moritz': 0}

In [60]:
demand_activities_from_hotel

{'taz_Bregaglia-taz_Bregaglia_Summer': 0,
 'taz_Bregaglia-taz_Sils_Engadin_Summer': 0,
 'taz_Bregaglia-taz_Silvaplana_Summer': 0,
 'taz_Bregaglia-taz_St_Moritz_Summer': 0,
 'taz_Bregaglia-taz_St_Moritz': 0,
 'taz_Bregaglia-taz_Bregaglia': 0,
 'taz_Bregaglia-taz_Silvaplana': 0,
 'taz_Bregaglia-taz_Sils_Engadin': 0,
 'taz_Bregaglia-taz_St_Moritz_Sink': 0,
 'taz_Silvaplana-taz_Bregaglia_Summer': 0,
 'taz_Silvaplana-taz_Sils_Engadin_Summer': 0,
 'taz_Silvaplana-taz_Silvaplana_Summer': 0,
 'taz_Silvaplana-taz_St_Moritz_Summer': 0,
 'taz_Silvaplana-taz_St_Moritz': 0,
 'taz_Silvaplana-taz_Bregaglia': 0,
 'taz_Silvaplana-taz_Silvaplana': 0,
 'taz_Silvaplana-taz_Sils_Engadin': 0,
 'taz_Silvaplana-taz_St_Moritz_Sink': 0,
 'taz_Sils_Engadin-taz_Bregaglia_Summer': 0,
 'taz_Sils_Engadin-taz_Sils_Engadin_Summer': 0,
 'taz_Sils_Engadin-taz_Silvaplana_Summer': 0,
 'taz_Sils_Engadin-taz_St_Moritz_Summer': 0,
 'taz_Sils_Engadin-taz_St_Moritz': 0,
 'taz_Sils_Engadin-taz_Bregaglia': 0,
 'taz_Sils_Engadin-

In [61]:
demand_activities_to_hotel

{'taz_Bregaglia-taz_Bregaglia_Summer': 0,
 'taz_Bregaglia-taz_Sils_Engadin_Summer': 0,
 'taz_Bregaglia-taz_Silvaplana_Summer': 0,
 'taz_Bregaglia-taz_St_Moritz_Summer': 0,
 'taz_Bregaglia-taz_St_Moritz': 0,
 'taz_Bregaglia-taz_Bregaglia': 0,
 'taz_Bregaglia-taz_Silvaplana': 0,
 'taz_Bregaglia-taz_Sils_Engadin': 0,
 'taz_Bregaglia-taz_St_Moritz_Sink': 0,
 'taz_Silvaplana-taz_Bregaglia_Summer': 0,
 'taz_Silvaplana-taz_Sils_Engadin_Summer': 0,
 'taz_Silvaplana-taz_Silvaplana_Summer': 0,
 'taz_Silvaplana-taz_St_Moritz_Summer': 0,
 'taz_Silvaplana-taz_St_Moritz': 0,
 'taz_Silvaplana-taz_Bregaglia': 0,
 'taz_Silvaplana-taz_Silvaplana': 0,
 'taz_Silvaplana-taz_Sils_Engadin': 0,
 'taz_Silvaplana-taz_St_Moritz_Sink': 0,
 'taz_Sils_Engadin-taz_Bregaglia_Summer': 0,
 'taz_Sils_Engadin-taz_Sils_Engadin_Summer': 0,
 'taz_Sils_Engadin-taz_Silvaplana_Summer': 0,
 'taz_Sils_Engadin-taz_St_Moritz_Summer': 0,
 'taz_Sils_Engadin-taz_St_Moritz': 0,
 'taz_Sils_Engadin-taz_Bregaglia': 0,
 'taz_Sils_Engadin-

In [62]:
for persontrip_child in list(persontrips_file_xmlroot):
    persontrip_in_routes = False
    for trip_child in list(routes_file_xmlroot):
        if trip_child.get("id") == persontrip_child.get("id"):
            persontrip_in_routes = True
            break
    
    if persontrip_in_routes == False:
        persontrips_file_xmlroot.remove(persontrip_child)

In [ ]:
persontrips_file_xmltree.write(persontrips_filepath_output)
routes_file_xmltree.write(routes_filepath_output)